In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")
embeddings_dir = os.getenv("EMBEDDINGS_DIR")

# Load the unified TSV file with image hashes
annotations_file = os.path.join(data_dir, "preprocessed_annotations.tsv")

In [ ]:
df = pd.read_csv(annotations_file, sep="\t")

In [ ]:
df.head()

In [ ]:
# Set device: GPU if available, otherwise CPU
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def generate_bert_embeddings(df, text_column, model_name="bert-base-uncased", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="BERT embeddings"):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # [CLS] token
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


def generate_roberta_embeddings(df, text_column, model_name="roberta-base", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="RoBERTa embeddings", disable=True):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


def generate_bertweet_embeddings(df, text_column, model_name="vinai/bertweet-base", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="BERTweet embeddings"):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


In [ ]:
os.makedirs(embeddings_dir, exist_ok=True)

bert_vecs = generate_bert_embeddings(df, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "embeddings_bert.npy"), bert_vecs)

print(f"{len(bert_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(bert_vecs[0])}")

In [ ]:
roberta_vecs = generate_roberta_embeddings(df, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "embeddings_roberta.npy"), roberta_vecs)

print(f"{len(bert_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(bert_vecs[0])}")

In [ ]:
bertweet_vecs = generate_bertweet_embeddings(df, "cleaned_text_bertweet")
np.save(os.path.join(embeddings_dir, "embeddings_bertweet.npy"), bertweet_vecs)

print(f"{len(bert_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(bert_vecs[0])}")